## Import libraries

In [2]:
from pathlib import Path
import json
import numpy as np

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

## Project Paths

In [3]:
PROJECT_PATH = Path.cwd().parent

CHUNKS_FILE = (PROJECT_PATH / "data" / "processed" / "chunks" / "chunks.jsonl")

EMBEDDINGS_FILE = (PROJECT_PATH / "data" / "processed" / "embeddings" / "embeddings.npy")

QDRANT_PATH = (PROJECT_PATH / "data" / "processed" / "vector_store" / "qdrant")

VECTOR_STORE_DIR = (PROJECT_PATH / "data" / "processed" / "vector_store" / "qdrant")

COLLECTION_NAME = "financial_policies"

print("Chunks:", CHUNKS_FILE)
print("Embeddings:", EMBEDDINGS_FILE)
print("Qdrant:", VECTOR_STORE_DIR)

Chunks: /Users/pushkarkamat/Desktop/financial-rag/data/processed/chunks/chunks.jsonl
Embeddings: /Users/pushkarkamat/Desktop/financial-rag/data/processed/embeddings/embeddings.npy
Qdrant: /Users/pushkarkamat/Desktop/financial-rag/data/processed/vector_store/qdrant


## Load Chunks and Embeddings

In [4]:
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    chunks = [json.loads(line) for line in f]

embeddings = np.load(EMBEDDINGS_FILE)

print("Number of Chunks:", len(chunks))
print("Embedding Shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Number of Chunks: 296
Embedding Shape: (296, 1024)
Embedding dtype: float32


## Validate Chunk-Embedding Alignment

In [5]:
assert len(chunks) == len(embeddings), (
    f"Mismatch: {len(chunks)} chunks vs {len(embeddings)} embeddings"
)

assert embeddings.shape[1] == 1024
print("Chunk and embedding alignment validate")

Chunk and embedding alignment validate



## Initialize Qdrant

In [6]:
QDRANT_PATH.mkdir(parents=True, exist_ok=True)

client = QdrantClient(path=str(QDRANT_PATH))

print("Qdrant client initialized.")
print("Storage path:", QDRANT_PATH)

Qdrant client initialized.
Storage path: /Users/pushkarkamat/Desktop/financial-rag/data/processed/vector_store/qdrant


## Create Qdrant Collection

In [7]:
existing_collections = [
    collection.name
    for collection in client.get_collections().collections
]

if COLLECTION_NAME in existing_collections:
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embeddings.shape[1],
        distance=Distance.COSINE,
    ),
)

print(f"Collection created: {COLLECTION_NAME}")

Collection created: financial_policies


## Prepare Qdrant Points

In [8]:
points = []

for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    payload = {
    "text": chunk["text"],
    **chunk["metadata"],
    }

    points.append(
        PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload=payload,
        )
    )

print("Points prepared:", len(points))

Points prepared: 296


In [9]:
for i in range(3):
    print("=" * 60)
    print("Point ID:", points[i].id)
    print("Vector first 5:", points[i].vector[:5])
    print("Vector norm:", np.linalg.norm(points[i].vector))

Point ID: 0
Vector first 5: [-0.012259459123015404, -0.018874801695346832, -5.352244625100866e-05, 0.01048474945127964, 0.01945601776242256]
Vector norm: 1.0000000251732992
Point ID: 1
Vector first 5: [0.014303087256848812, -0.021001998335123062, 0.020123904570937157, 0.020624032244086266, 0.009284825064241886]
Vector norm: 1.0000000418031523
Point ID: 2
Vector first 5: [0.0225226953625679, -0.01994173415005207, 0.04215560108423233, 0.01388160977512598, -0.0037593606393784285]
Vector norm: 1.0000000192146847


In [10]:
for i in range(3):
    print("=" * 60)
    print("Point ID:", i)
    print("Embedding first 5:", embeddings[i][:5])
    print("Embedding norm:", np.linalg.norm(embeddings[i]))

Point ID: 0
Embedding first 5: [-1.2259459e-02 -1.8874802e-02 -5.3522446e-05  1.0484749e-02
  1.9456018e-02]
Embedding norm: 1.0
Point ID: 1
Embedding first 5: [ 0.01430309 -0.021002    0.0201239   0.02062403  0.00928483]
Embedding norm: 1.0
Point ID: 2
Embedding first 5: [ 0.0225227  -0.01994173  0.0421556   0.01388161 -0.00375936]
Embedding norm: 1.0


## Upload Vectors to Qdrant

In [11]:
operation_info = client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,
)

print("Upload completed.")
print(operation_info)

Upload completed.
operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>


## Validate Qdrant Collection

In [12]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Vectors stored:", collection_info.points_count)

assert collection_info.points_count == len(chunks)

print("Qdrant validation passed.")

Collection: financial_policies
Vectors stored: 296
Qdrant validation passed.


In [13]:
stored_points = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[0, 1, 2],
    with_payload=True,
    with_vectors=True,
)

for point in stored_points:
    vector = np.array(point.vector)

    print("=" * 60)
    print("ID:", point.id)
    print("First 5 values:", vector[:5])
    print("Norm:", np.linalg.norm(vector))
    print("Chunk ID:", point.payload.get("chunk_id"))

ID: 0
First 5 values: [-1.22594591e-02 -1.88748017e-02 -5.35224463e-05  1.04847495e-02
  1.94560178e-02]
Norm: 1.0000000251732992
Chunk ID: chunk_00000000
ID: 1
First 5 values: [ 0.01430309 -0.021002    0.0201239   0.02062403  0.00928483]
Norm: 1.0000000418031523
Chunk ID: chunk_00000001
ID: 2
First 5 values: [ 0.0225227  -0.01994173  0.0421556   0.01388161 -0.00375936]
Norm: 1.0000000192146847
Chunk ID: chunk_00000002


In [14]:
client.close()
print("Qdrant client closed.")

Qdrant client closed.
